In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from scipy.optimize import minimize

DATA_DIR = Path("../data/mock")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

products = pd.read_csv(DATA_DIR / "products.csv")
preferences = pd.read_csv(DATA_DIR / "user_preferences.csv")
c1 = pd.read_csv(DATA_DIR / "c1_product_aspects.csv")
c2 = pd.read_csv(DATA_DIR / "c2_review_evidence.csv")
c3 = pd.read_csv(DATA_DIR / "c3_temporal_features.csv")

print("Data loaded successfully.")

Data loaded successfully.


In [2]:
features = c1.merge(
    c3,
    on=["product_id", "aspect"],
    how="inner"
)

features["nominal_aspect_score"] = (
    features["aspect_quality"]
    + features["recent_quality"]
) / 2

aspect_order = preferences["aspect"].tolist()

aspect_matrix = (
    features
    .pivot(
        index="product_id",
        columns="aspect",
        values="nominal_aspect_score"
    )
    [aspect_order]
)

max_budget = preferences["max_budget"].iloc[0]

eligible_ids = products[
    products["price"] <= max_budget
]["product_id"]

X = aspect_matrix.loc[
    aspect_matrix.index.isin(eligible_ids)
].copy()

display(X)

aspect,battery,camera,performance,display,durability,price
product_id,,,,,,
P001,0.890,0.735,0.905,0.815,0.785,0.695
P002,0.735,0.900,0.830,0.855,0.750,0.820
P004,0.795,0.675,0.725,0.740,0.775,0.915
P005,0.875,0.785,0.815,0.795,0.905,0.775


In [3]:
pref = preferences[
    ["aspect", "importance"]
].copy()

pref["weight"] = (
    pref["importance"]
    / pref["importance"].sum()
)

original_weights = (
    pref
    .set_index("aspect")
    .loc[aspect_order, "weight"]
    .to_numpy()
)

display(pref)

print(
    "Weight sum:",
    original_weights.sum()
)

,aspect,importance,weight
0,battery,5,0.238095
1,camera,3,0.142857
2,performance,4,0.190476
3,display,3,0.142857
4,durability,2,0.095238
5,price,4,0.190476


Weight sum: 0.9999999999999999


In [4]:
def rank_with_weights(
    X,
    weights
):
    scores = np.dot(
        X.to_numpy(),
        weights
    )

    ranking = pd.DataFrame({
        "product_id": X.index,
        "score": scores
    })

    ranking = ranking.merge(
        products[
            ["product_id", "product_name", "price"]
        ],
        on="product_id",
        how="left"
    )

    ranking = (
        ranking
        .sort_values(
            "score",
            ascending=False
        )
        .reset_index(drop=True)
    )

    ranking["rank"] = (
        np.arange(1, len(ranking) + 1)
    )

    return ranking


original_ranking = rank_with_weights(
    X,
    original_weights
)

display(original_ranking)

,product_id,score,product_name,price,rank
0,P005,0.823095,Phone E,160000,1
1,P001,0.812857,Phone A,165000,2
2,P002,0.811429,Phone B,150000,3
3,P004,0.777619,Phone D,140000,4


In [5]:
TARGET_PRODUCT = "P002"
MARGIN = 0.0001

target_name = products.loc[
    products["product_id"] == TARGET_PRODUCT,
    "product_name"
].iloc[0]

print(
    "Counterfactual target:",
    target_name,
    "(" + TARGET_PRODUCT + ")"
)

Counterfactual target: Phone B (P002)


In [6]:
def objective(new_weights):
    return np.sum(
        (new_weights - original_weights) ** 2
    )

In [7]:
target_vector = (
    X.loc[TARGET_PRODUCT]
    .to_numpy()
)

constraints = [
    {
        "type": "eq",
        "fun": lambda w:
            np.sum(w) - 1.0
    }
]

for competitor_id in X.index:

    if competitor_id == TARGET_PRODUCT:
        continue

    competitor_vector = (
        X.loc[competitor_id]
        .to_numpy()
    )

    constraints.append({
        "type": "ineq",
        "fun": lambda w,
        t=target_vector,
        c=competitor_vector:
            np.dot(t - c, w) - MARGIN
    })

In [8]:
result = minimize(
    objective,
    x0=original_weights,
    method="SLSQP",
    bounds=[
        (0.0, 1.0)
        for _ in aspect_order
    ],
    constraints=constraints,
    options={
        "maxiter": 2000,
        "ftol": 1e-12
    }
)

print(
    "Optimization success:",
    result.success
)

print(
    "Message:",
    result.message
)

Optimization success: True
Message: Optimization terminated successfully


In [9]:
counterfactual_weights = result.x

counterfactual_table = pd.DataFrame({
    "aspect": aspect_order,
    "original_weight": original_weights,
    "counterfactual_weight":
        counterfactual_weights
})

counterfactual_table["change"] = (
    counterfactual_table[
        "counterfactual_weight"
    ]
    - counterfactual_table[
        "original_weight"
    ]
)

counterfactual_table["absolute_change"] = (
    counterfactual_table[
        "change"
    ].abs()
)

counterfactual_table = (
    counterfactual_table
    .sort_values(
        "absolute_change",
        ascending=False
    )
)

display(counterfactual_table)

,aspect,original_weight,counterfactual_weight,change,absolute_change
4,durability,0.095238,0.067764,-0.027475,0.027475
0,battery,0.238095,0.213463,-0.024632,0.024632
1,camera,0.142857,0.166542,0.023685,0.023685
3,display,0.142857,0.156121,0.013264,0.013264
5,price,0.190476,0.200898,0.010421,0.010421
2,performance,0.190476,0.195213,0.004737,0.004737


In [10]:
counterfactual_ranking = (
    rank_with_weights(
        X,
        counterfactual_weights
    )
)

display(counterfactual_ranking)

print(
    "Counterfactual top product:",
    counterfactual_ranking.iloc[0][
        "product_name"
    ]
)

,product_id,score,product_name,price,rank
0,P002,0.817852,Phone B,150000,1
1,P005,0.817752,Phone E,160000,2
2,P001,0.809115,Phone A,165000,3
3,P004,0.775516,Phone D,140000,4


Counterfactual top product: Phone B


In [11]:
l1_change = np.sum(
    np.abs(
        counterfactual_weights
        - original_weights
    )
)

l2_change = np.sqrt(
    np.sum(
        (
            counterfactual_weights
            - original_weights
        ) ** 2
    )
)

changed_aspects = (
    counterfactual_table[
        counterfactual_table[
            "absolute_change"
        ] > 1e-6
    ]
)

print(
    "L1 preference change:",
    round(l1_change, 6)
)

print(
    "L2 preference change:",
    round(l2_change, 6)
)

print(
    "Number of changed aspects:",
    len(changed_aspects)
)

L1 preference change: 0.104214
L2 preference change: 0.047218
Number of changed aspects: 6


In [12]:
assert result.success

assert np.isclose(
    counterfactual_weights.sum(),
    1.0
)

assert (
    counterfactual_weights >= 0
).all()

assert (
    counterfactual_ranking.iloc[0][
        "product_id"
    ]
    == TARGET_PRODUCT
)

print(
    "COUNTERFACTUAL VALIDITY TEST: PASSED"
)

COUNTERFACTUAL VALIDITY TEST: PASSED


In [16]:
import sys

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent

sys.path.insert(
    0,
    str(ROOT)
)

from component4.src.evidence_uncertainty import (
    weighted_bootstrap_evidence
)

In [17]:
N_BOOTSTRAP = 500

evidence_samples = (
    weighted_bootstrap_evidence(
        c2,
        n_bootstrap=N_BOOTSTRAP,
        random_state=42
    )
)

evidence_samples = evidence_samples.rename(
    columns={
        "bootstrap_id":
        "simulation_id"
    }
)

print(
    "Evidence rows:",
    len(evidence_samples)
)

Evidence rows: 15000


In [18]:
nominal_evidence = (
    c2.groupby(
        ["product_id", "aspect"]
    )
    .apply(
        lambda g:
            np.average(
                g["evidence_quality"],
                weights=g["evidence_quality"]
            )
    )
    .reset_index(
        name="nominal_evidence_reliability"
    )
)

display(
    nominal_evidence.head()
)

,product_id,aspect,nominal_evidence_reliability
0,P001,battery,0.849841
1,P001,camera,0.783793
2,P001,display,0.789660
3,P001,durability,0.792298
4,P001,performance,0.865564


In [19]:
robust_data = features[
    [
        "product_id",
        "aspect",
        "nominal_aspect_score"
    ]
].merge(
    nominal_evidence,
    on=[
        "product_id",
        "aspect"
    ],
    how="inner"
)

robust_data = robust_data.merge(
    evidence_samples,
    on=[
        "product_id",
        "aspect"
    ],
    how="inner"
)

robust_data["evidence_factor"] = (
    robust_data[
        "evidence_reliability"
    ]
    /
    robust_data[
        "nominal_evidence_reliability"
    ]
)

robust_data["adjusted_score"] = np.clip(
    robust_data["nominal_aspect_score"]
    * robust_data["evidence_factor"],
    0,
    1
)

In [20]:
cf_weight_map = dict(
    zip(
        aspect_order,
        counterfactual_weights
    )
)

robust_data["cf_weight"] = (
    robust_data["aspect"]
    .map(cf_weight_map)
)

robust_data[
    "weighted_contribution"
] = (
    robust_data[
        "adjusted_score"
    ]
    * robust_data[
        "cf_weight"
    ]
)

In [21]:
robust_scores = (
    robust_data
    .groupby(
        [
            "simulation_id",
            "product_id"
        ],
        as_index=False
    )
    ["weighted_contribution"]
    .sum()
    .rename(
        columns={
            "weighted_contribution":
            "score"
        }
    )
)

robust_scores = robust_scores.merge(
    products,
    on="product_id",
    how="left"
)

robust_scores = robust_scores[
    robust_scores["price"]
    <= max_budget
].copy()

robust_scores = (
    robust_scores
    .sort_values(
        [
            "simulation_id",
            "score"
        ],
        ascending=[
            True,
            False
        ]
    )
)

robust_scores["rank"] = (
    robust_scores
    .groupby("simulation_id")
    .cumcount() + 1
)

In [22]:
robust_top1 = robust_scores[
    robust_scores["rank"] == 1
]

robust_validity = (
    robust_top1["product_id"]
    .eq(TARGET_PRODUCT)
    .mean()
)

print(
    "Robust Validity:",
    round(
        robust_validity,
        4
    )
)

print(
    "Target remained Rank 1 in:",
    int(
        (
            robust_top1[
                "product_id"
            ]
            == TARGET_PRODUCT
        ).sum()
    ),
    "out of",
    N_BOOTSTRAP,
    "evidence simulations"
)

Robust Validity: 0.408
Target remained Rank 1 in: 204 out of 500 evidence simulations


In [23]:
counterfactual_table.to_csv(
    OUTPUT_DIR /
    "counterfactual_preference_changes.csv",
    index=False
)

counterfactual_ranking.to_csv(
    OUTPUT_DIR /
    "counterfactual_ranking.csv",
    index=False
)

counterfactual_metrics = pd.DataFrame({
    "metric": [
        "Validity",
        "L1 Minimality",
        "L2 Minimality",
        "Robust Validity"
    ],

    "value": [
        1.0,
        l1_change,
        l2_change,
        robust_validity
    ]
})

counterfactual_metrics.to_csv(
    OUTPUT_DIR /
    "counterfactual_metrics.csv",
    index=False
)

print(
    "Counterfactual outputs saved."
)

Counterfactual outputs saved.


In [24]:
holdout_evidence_samples = weighted_bootstrap_evidence(
    c2,
    n_bootstrap=500,
    random_state=123
).rename(
    columns={
        "bootstrap_id": "simulation_id"
    }
)

print(
    "Holdout evidence rows:",
    len(holdout_evidence_samples)
)

Holdout evidence rows: 15000


In [25]:
def build_evidence_scenarios(
    features,
    evidence_samples,
    nominal_evidence,
    products,
    max_budget
):
    data = features[
        [
            "product_id",
            "aspect",
            "nominal_aspect_score"
        ]
    ].merge(
        nominal_evidence,
        on=["product_id", "aspect"],
        how="inner"
    )

    data = data.merge(
        evidence_samples,
        on=["product_id", "aspect"],
        how="inner"
    )

    data["evidence_factor"] = (
        data["evidence_reliability"]
        /
        data["nominal_evidence_reliability"]
    )

    data["adjusted_score"] = np.clip(
        data["nominal_aspect_score"]
        * data["evidence_factor"],
        0,
        1
    )

    eligible = products[
        products["price"] <= max_budget
    ]["product_id"]

    return data[
        data["product_id"].isin(eligible)
    ].copy()

In [26]:
train_scenarios = build_evidence_scenarios(
    features,
    evidence_samples,
    nominal_evidence,
    products,
    max_budget
)

holdout_scenarios = build_evidence_scenarios(
    features,
    holdout_evidence_samples,
    nominal_evidence,
    products,
    max_budget
)

print(
    "Train simulations:",
    train_scenarios["simulation_id"].nunique()
)

print(
    "Holdout simulations:",
    holdout_scenarios["simulation_id"].nunique()
)

Train simulations: 500
Holdout simulations: 500


In [27]:
def evaluate_robust_validity(
    scenario_data,
    weights,
    target_product
):
    weight_map = dict(
        zip(
            aspect_order,
            weights
        )
    )

    data = scenario_data.copy()

    data["weight"] = (
        data["aspect"].map(weight_map)
    )

    data["contribution"] = (
        data["adjusted_score"]
        * data["weight"]
    )

    scores = (
        data.groupby(
            ["simulation_id", "product_id"],
            as_index=False
        )["contribution"]
        .sum()
    )

    winners = (
        scores.sort_values(
            ["simulation_id", "contribution"],
            ascending=[True, False]
        )
        .groupby("simulation_id")
        .first()
        .reset_index()
    )

    validity = (
        winners["product_id"]
        .eq(target_product)
        .mean()
    )

    return validity

In [28]:
standard_train_validity = evaluate_robust_validity(
    train_scenarios,
    counterfactual_weights,
    TARGET_PRODUCT
)

standard_holdout_validity = evaluate_robust_validity(
    holdout_scenarios,
    counterfactual_weights,
    TARGET_PRODUCT
)

print(
    "Standard CF train validity:",
    round(standard_train_validity, 4)
)

print(
    "Standard CF holdout validity:",
    round(standard_holdout_validity, 4)
)

Standard CF train validity: 0.408
Standard CF holdout validity: 0.384


In [29]:
def scenario_target_margins(
    scenario_data,
    weights,
    target_product
):
    weight_map = dict(
        zip(
            aspect_order,
            weights
        )
    )

    data = scenario_data.copy()

    data["weight"] = (
        data["aspect"].map(weight_map)
    )

    data["score_part"] = (
        data["adjusted_score"]
        * data["weight"]
    )

    scores = (
        data.groupby(
            ["simulation_id", "product_id"],
            as_index=False
        )["score_part"]
        .sum()
    )

    target_scores = (
        scores[
            scores["product_id"]
            == target_product
        ]
        .set_index("simulation_id")[
            "score_part"
        ]
    )

    competitor_scores = (
        scores[
            scores["product_id"]
            != target_product
        ]
        .groupby("simulation_id")[
            "score_part"
        ]
        .max()
    )

    margins = (
        target_scores
        - competitor_scores
    )

    return margins.to_numpy()

In [30]:
ROBUST_MARGIN = 0.0001


def robust_objective(
    weights,
    penalty_strength
):
    # Keep preferences close to original user profile
    distance = np.sum(
        (
            weights
            - original_weights
        ) ** 2
    )

    margins = scenario_target_margins(
        train_scenarios,
        weights,
        TARGET_PRODUCT
    )

    violations = np.maximum(
        0,
        ROBUST_MARGIN - margins
    )

    robustness_penalty = np.mean(
        violations ** 2
    )

    return (
        distance
        + penalty_strength
        * robustness_penalty
    )

In [31]:
robust_constraints = [
    {
        "type": "eq",
        "fun": lambda w:
            np.sum(w) - 1.0
    }
]

for competitor_id in X.index:

    if competitor_id == TARGET_PRODUCT:
        continue

    competitor_vector = (
        X.loc[competitor_id]
        .to_numpy()
    )

    robust_constraints.append({
        "type": "ineq",
        "fun": lambda w,
        t=target_vector,
        c=competitor_vector:
            np.dot(t - c, w) - MARGIN
    })

In [32]:
penalty_values = [
    1,
    10,
    100,
    1000,
    10000,
    100000
]

robust_candidates = []

for penalty in penalty_values:

    result_robust = minimize(
        lambda w:
            robust_objective(
                w,
                penalty
            ),
        x0=counterfactual_weights,
        method="SLSQP",
        bounds=[
            (0.0, 1.0)
            for _ in aspect_order
        ],
        constraints=robust_constraints,
        options={
            "maxiter": 3000,
            "ftol": 1e-12
        }
    )

    weights_candidate = result_robust.x

    train_validity = evaluate_robust_validity(
        train_scenarios,
        weights_candidate,
        TARGET_PRODUCT
    )

    holdout_validity = evaluate_robust_validity(
        holdout_scenarios,
        weights_candidate,
        TARGET_PRODUCT
    )

    l2 = np.linalg.norm(
        weights_candidate
        - original_weights
    )

    robust_candidates.append({
        "penalty": penalty,
        "success": result_robust.success,
        "train_robust_validity":
            train_validity,
        "holdout_robust_validity":
            holdout_validity,
        "l2_change": l2,
        "weights": weights_candidate
    })

In [33]:
candidate_summary = pd.DataFrame([
    {
        "penalty": c["penalty"],
        "success": c["success"],
        "train_robust_validity":
            c["train_robust_validity"],
        "holdout_robust_validity":
            c["holdout_robust_validity"],
        "l2_change":
            c["l2_change"]
    }
    for c in robust_candidates
])

display(candidate_summary)

,penalty,success,train_robust_validity,holdout_robust_validity,l2_change
0,1,True,0.408,0.386,0.047233
1,10,True,0.422,0.398,0.048454
2,100,True,0.682,0.658,0.125685
3,1000,True,0.892,0.878,0.221102
4,10000,True,0.954,0.964,0.313855
5,100000,True,0.968,0.984,0.398904


In [34]:
ROBUST_TARGET = 0.80

valid_candidates = [
    c for c in robust_candidates
    if (
        c["success"]
        and
        c["holdout_robust_validity"]
        >= ROBUST_TARGET
    )
]

if valid_candidates:

    best_robust = min(
        valid_candidates,
        key=lambda c:
            c["l2_change"]
    )

    target_reached = True

else:

    best_robust = max(
        robust_candidates,
        key=lambda c: (
            c["holdout_robust_validity"],
            -c["l2_change"]
        )
    )

    target_reached = False


robust_weights = best_robust["weights"]

print(
    "80% robust target reached:",
    target_reached
)

print(
    "Selected penalty:",
    best_robust["penalty"]
)

print(
    "Train Robust Validity:",
    round(
        best_robust[
            "train_robust_validity"
        ],
        4
    )
)

print(
    "Holdout Robust Validity:",
    round(
        best_robust[
            "holdout_robust_validity"
        ],
        4
    )
)

80% robust target reached: True
Selected penalty: 1000
Train Robust Validity: 0.892
Holdout Robust Validity: 0.878


In [35]:
robust_cf_table = pd.DataFrame({
    "aspect": aspect_order,
    "original_weight":
        original_weights,
    "standard_cf_weight":
        counterfactual_weights,
    "robust_cf_weight":
        robust_weights
})

robust_cf_table[
    "robust_change"
] = (
    robust_cf_table[
        "robust_cf_weight"
    ]
    -
    robust_cf_table[
        "original_weight"
    ]
)

robust_cf_table[
    "absolute_robust_change"
] = (
    robust_cf_table[
        "robust_change"
    ].abs()
)

robust_cf_table = (
    robust_cf_table
    .sort_values(
        "absolute_robust_change",
        ascending=False
    )
)

display(robust_cf_table)

,aspect,original_weight,standard_cf_weight,robust_cf_weight,robust_change,absolute_robust_change
0,battery,0.238095,0.213463,0.100139,-0.137957,0.137957
1,camera,0.142857,0.166542,0.242795,0.099938,0.099938
4,durability,0.095238,0.067764,0.000000,-0.095238,0.095238
3,display,0.142857,0.156121,0.216316,0.073459,0.073459
5,price,0.190476,0.200898,0.262872,0.072396,0.072396
2,performance,0.190476,0.195213,0.177878,-0.012598,0.012598


In [36]:
robust_ranking = rank_with_weights(
    X,
    robust_weights
)

display(robust_ranking)

print(
    "Robust CF nominal top product:",
    robust_ranking.iloc[0][
        "product_name"
    ]
)

,product_id,score,product_name,price,rank
0,P002,0.840262,Phone B,150000,1
1,P005,0.798883,Phone E,160000,2
2,P001,0.787551,Phone A,165000,3
3,P004,0.773060,Phone D,140000,4


Robust CF nominal top product: Phone B


In [37]:
standard_l2 = np.linalg.norm(
    counterfactual_weights
    - original_weights
)

robust_l2 = np.linalg.norm(
    robust_weights
    - original_weights
)

comparison = pd.DataFrame({
    "counterfactual": [
        "Standard CF",
        "Robust CF"
    ],

    "nominal_validity": [
        1.0,
        float(
            robust_ranking.iloc[0][
                "product_id"
            ]
            == TARGET_PRODUCT
        )
    ],

    "L2_minimality": [
        standard_l2,
        robust_l2
    ],

    "holdout_robust_validity": [
        standard_holdout_validity,
        best_robust[
            "holdout_robust_validity"
        ]
    ]
})

display(comparison)

,counterfactual,nominal_validity,L2_minimality,holdout_robust_validity
0,Standard CF,1.0,0.047218,0.384
1,Robust CF,1.0,0.221102,0.878


In [38]:
assert np.isclose(
    robust_weights.sum(),
    1.0
)

assert (
    robust_weights >= 0
).all()

assert (
    robust_ranking.iloc[0][
        "product_id"
    ]
    == TARGET_PRODUCT
)

assert (
    0
    <= best_robust[
        "holdout_robust_validity"
    ]
    <= 1
)

print(
    "ROBUST COUNTERFACTUAL TESTS: PASSED"
)

ROBUST COUNTERFACTUAL TESTS: PASSED


In [39]:
robust_cf_table.to_csv(
    OUTPUT_DIR /
    "robust_counterfactual_preferences.csv",
    index=False
)

comparison.to_csv(
    OUTPUT_DIR /
    "counterfactual_comparison.csv",
    index=False
)

candidate_summary.to_csv(
    OUTPUT_DIR /
    "robust_counterfactual_search.csv",
    index=False
)

print(
    "Robust counterfactual outputs saved."
)

Robust counterfactual outputs saved.
